# Notebook 7 — Encode Geospatial Network into the CANOE Schema

This notebook converts the geospatial outputs developed in previous notebooks into a CANOE-compatible SQLite database.

The objective is to replace the synthetic grid-neighbor representation used in the prototype model with a transport network derived from the Canadian basemap and road connectivity analysis while preserving compatibility with the existing CANOE/TEMOA model structure.

Transport links are represented using CANOE pseudo-regions of the form

```
region_from-region_to
```

where the two regions correspond to adjacent basemap polygons connected by existing infrastructure. This forms a dual graph representation of Canadian infrastructure layers aligned under a specified spatial resolution.

Initially, this notebook focuses on road-based transport technologies and encodes only links for which road connectivity has been identified. The absence of a link implies that transport between those regions is infeasible.

Rather than rebuilding the complete database from raw CSV files, this notebook loads the existing CANOE database and performs a schema reconciliation step. Inherited tables are filtered to the geospatial region topology and augmented with new transport technologies, producing a functional geospatial test database suitable for MILP execution.

The resulting database provides an intermediate development layer between the geospatial preprocessing workflow and eventual integration into the core CANOE modules.

---

## Inputs

### Basemap regions

From Notebook 4:

* Regional polygon geometries
* Region identifiers
* Region centroids

### Neighbor relationships

From Notebook 5:

* Polygon adjacency graph
* Neighbor pairs
* Inter-region distances

### Road connectivity

From Notebook 6:

* Weak road connectivity
* Strong road connectivity

### Existing CANOE database

* Existing CANOE SQLite database
* Schema definitions
* Technology definitions
* Commodity definitions
* Supporting tables

---

## Outputs

This notebook modifies and validates CANOE tables including:

* Region
* Technology
* Efficiency
* CostVariable
* CostInvest
* ETLSegment
* Demand
* LimitCapacity
* Supporting schema tables

and exports complete SQLite databases suitable for direct use by the CANOE/TEMOA solver.

---

## Conceptual workflow

1. Load the regional basemap and road connectivity outputs.
2. Load the existing CANOE database.
3. Replace the synthetic region representation with geospatial regions.
4. Build transport edges from connected neighboring regions.
5. Encode transport technologies for each valid edge.
6. Reconcile inherited node and edge tables with the geospatial topology.
7. Validate schema consistency.
8. Export SQLite databases.
9. Test MILP execution using the existing CANOE workflow.

This notebook serves as a graph-to-schema encoder and schema reconciliation layer between the geospatial preprocessing workflow and eventual integration into the core CANOE modules.

In [1]:
# =============================================================================
# Dependencies
# =============================================================================

from pathlib import Path

import sqlite3

import numpy as np
import pandas as pd

import geopandas as gpd

import matplotlib.pyplot as plt

import db_mgmt

In [2]:
# =============================================================================
# Project directories
# =============================================================================

PROJECT_ROOT = Path.cwd().parent

DATA_FILES = PROJECT_ROOT / "data_files"

RAW_BASEMAPS = DATA_FILES / "raw" / "basemaps"

PROCESSED_BASEMAPS = DATA_FILES / "processed" / "basemaps"
PROCESSED_GRAPH = DATA_FILES / "processed" / "graph"
PROCESSED_ROAD_CONNECTIVITY = DATA_FILES / "processed" / "road_connectivity"
PROCESSED_SCHEMA = DATA_FILES / "processed" / "schema"

PROCESSED_SCHEMA.mkdir(
    parents=True,
    exist_ok=True,
)

In [3]:
# =============================================================================
# Input files
# =============================================================================

RAW_BASEMAP_PATH = RAW_BASEMAPS / "lpr_000b21a_e.shp"

BASEMAP_CENTROID_PATH = PROCESSED_BASEMAPS / "canada_basemap_1deg_centroid.gpkg"
BASEMAP_INTERSECTS_PATH = PROCESSED_BASEMAPS / "canada_basemap_1deg_intersects.gpkg"

NEIGHBOR_GRAPH_PATH = PROCESSED_GRAPH / "canada_basemap_1deg_centroid_neighbors.gpkg"

ROAD_EDGES_WEAK_PATH = (
    PROCESSED_ROAD_CONNECTIVITY
    / "CANADA_filtered_road_networks__canada_basemap_1deg_centroid_weak_road_edges.gpkg"
)

RAW_SCHEMA_PATH = DATA_FILES / "canoe_dataset_schema.sql"

BASELINE_SQLITE_PATH = DATA_FILES / "CANOE_geospatial.sqlite"

In [4]:
# =============================================================================
# Output files
# =============================================================================

OUTPUT_SQLITE_WEAK_PATH = (
    PROCESSED_SCHEMA
    / "CANOE_geospatial_1deg_roads_weak.sqlite"
)

OUTPUT_SQLITE_STRONG_PATH = (
    PROCESSED_SCHEMA
    / "CANOE_geospatial_1deg_roads_strong.sqlite"
)

In [5]:
# =============================================================================
# Validate input files
# =============================================================================

input_paths = {
    "raw_basemap": RAW_BASEMAP_PATH,
    "basemap_centroid": BASEMAP_CENTROID_PATH,
    "basemap_intersects": BASEMAP_INTERSECTS_PATH,
    "neighbor_graph": NEIGHBOR_GRAPH_PATH,
    "road_edges_weak": ROAD_EDGES_WEAK_PATH,
    "raw_schema": RAW_SCHEMA_PATH,
    "baseline_sqlite": BASELINE_SQLITE_PATH,
}

missing_paths = {
    name: path
    for name, path in input_paths.items()
    if not path.exists()
}

if missing_paths:
    for name, path in missing_paths.items():
        print(f"Missing {name}: {path}")
    raise FileNotFoundError("One or more required input files are missing.")

print("All required input files found.")

All required input files found.


In [6]:
# =============================================================================
# Load geospatial inputs
# =============================================================================

regions = gpd.read_file(BASEMAP_CENTROID_PATH)
regions_intersects = gpd.read_file(BASEMAP_INTERSECTS_PATH)
neighbor_graph = gpd.read_file(NEIGHBOR_GRAPH_PATH)
road_edges_weak = gpd.read_file(ROAD_EDGES_WEAK_PATH)

print(f"Regions centroid layer: {len(regions):,} rows")
print(f"Regions intersects layer: {len(regions_intersects):,} rows")
print(f"Neighbor graph: {len(neighbor_graph):,} rows")
print(f"Weak road edges: {len(road_edges_weak):,} rows")

print("\nCRS:")
print(f"regions: {regions.crs}")
print(f"regions_intersects: {regions_intersects.crs}")
print(f"neighbor_graph: {neighbor_graph.crs}")
print(f"road_edges_weak: {road_edges_weak.crs}")

Regions centroid layer: 1,692 rows
Regions intersects layer: 2,276 rows
Neighbor graph: 1,692 rows
Weak road edges: 1,518 rows

CRS:
regions: EPSG:4326
regions_intersects: EPSG:4326
neighbor_graph: EPSG:4326
road_edges_weak: EPSG:4326


In [7]:
# =============================================================================
# Load baseline CANOE database
# =============================================================================

db = db_mgmt.sqlite_to_dfs(
    BASELINE_SQLITE_PATH,
)

print(f"{len(db)} tables loaded.")

85 tables loaded.


In [8]:
# =============================================================================
# Inspect available tables
# =============================================================================

sorted(db.keys())

['CapacityCredit',
 'CapacityFactorProcess',
 'CapacityFactorTech',
 'CapacityToActivity',
 'Commodity',
 'CommodityType',
 'ConstructionInput',
 'CostEmission',
 'CostFixed',
 'CostInvest',
 'CostVariable',
 'DataQualityCredibility',
 'DataQualityGeography',
 'DataQualityStructure',
 'DataQualityTechnology',
 'DataQualityTime',
 'DataSet',
 'DataSource',
 'Demand',
 'DemandSpecificDistribution',
 'ETLSegment',
 'Efficiency',
 'EfficiencyVariable',
 'EmissionActivity',
 'EmissionEmbodied',
 'EmissionEndOfLife',
 'EndOfLifeOutput',
 'ExistingCapacity',
 'LifetimeProcess',
 'LifetimeSurvivalCurve',
 'LifetimeTech',
 'LimitActivity',
 'LimitActivityShare',
 'LimitAnnualCapacityFactor',
 'LimitCapacity',
 'LimitCapacityShare',
 'LimitDegrowthCapacity',
 'LimitDegrowthNewCapacity',
 'LimitDegrowthNewCapacityDelta',
 'LimitEmission',
 'LimitGrowthCapacity',
 'LimitGrowthNewCapacity',
 'LimitGrowthNewCapacityDelta',
 'LimitNewCapacity',
 'LimitNewCapacityShare',
 'LimitResource',
 'LimitSeaso

In [9]:
# =============================================================================
# Inspect core schema tables
# =============================================================================

core_tables = [
    "Region",
    "Technology",
    "TechnologyType",
    "Commodity",
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
    "LimitCapacity",
    "LimitNewCapacity",
    "ExistingCapacity",
    "DataSet",
    "DataSource",
]

for table in core_tables:
    df = db[table]
    print(f"\n{table}")
    print(f"  rows: {len(df):,}")
    print(f"  columns: {list(df.columns)}")


Region
  rows: 2,259
  columns: ['region', 'notes']

Technology
  rows: 12
  columns: ['tech', 'flag', 'sector', 'category', 'sub_category', 'unlim_cap', 'annual', 'reserve', 'curtail', 'retire', 'flex', 'exchange', 'seas_stor', 'description', 'data_id']

TechnologyType
  rows: 4
  columns: ['label', 'description']

Commodity
  rows: 7
  columns: ['name', 'flag', 'description', 'data_id']

Efficiency
  rows: 62,188
  columns: ['region', 'input_comm', 'tech', 'vintage', 'output_comm', 'efficiency', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']

CostVariable
  rows: 50,487
  columns: ['region', 'period', 'tech', 'vintage', 'cost', 'units', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']

CostInvest
  rows: 4,518
  columns: ['region', 'tech', 'vintage', 'cost', 'units', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']

ETLSegment
  rows: 193,552
  columns: ['r

In [10]:
# =============================================================================
# Inspect baseline technologies and commodities
# =============================================================================

display(
    db["Technology"].sort_values("tech")
)

display(
    db["TechnologyType"].sort_values("label")
)

display(
    db["Commodity"].sort_values("name")
)

,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
2,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
7,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
0,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
5,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
11,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
10,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
9,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
6,H2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
1,H2_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001


,label,description
2,p,production
0,pb,baseload production technology
1,ps,storage production technology
3,t,transport


,name,flag,description,data_id
1,ch3oh,wa,methanol,None
2,co2,wa,co2 captured,None
5,d_gsl,d,gasoline demand,None
4,elc,wa,electricity,None
6,ethos,s,dummy,None
3,gsl,wa,gasoline,None
0,h2,wa,hydrogen,None


In [11]:
# =============================================================================
# Inspect baseline technology definitions
# =============================================================================

technology = db["Technology"].copy()

display(
    technology.sort_values(
        "tech"
    ).reset_index(
        drop=True
    )
)

,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
0,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
1,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
2,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
3,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
5,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
6,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
7,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
8,H2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
9,H2_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001


In [12]:
# =============================================================================
# Inspect baseline regions
# =============================================================================

region = db["Region"].copy()

print(f"Number of regions: {len(region):,}")

display(
    region.head(20)
)

Number of regions: 2,259


,region,notes
0,R0,None
1,R1,None
2,R2,None
3,R3,None
4,R4,None
5,R5,None
6,R6,None
7,R7,None
8,R8,None
9,R9,None


In [13]:
# =============================================================================
# Inspect geospatial region identifiers
# =============================================================================

print(f"Centroid regions: {len(regions):,}")
print(f"Intersect regions: {len(regions_intersects):,}")

display(
    regions.head()
)

Centroid regions: 1,692
Intersect regions: 2,276


,lon_min,lon_max,lat_min,lat_max,lon,lat,region,site_id,resolution_deg,keep_method,geometry
0,-82.0,-81.0,43.0,44.0,-81.5,43.5,R0,R0,1.0,centroid,"POLYGON ((-81 43, -81 44, -82 44, -82 43, -81 ..."
1,-81.0,-80.0,43.0,44.0,-80.5,43.5,R1,R1,1.0,centroid,"POLYGON ((-80 43, -80 44, -81 44, -81 43, -80 ..."
2,-66.0,-65.0,43.0,44.0,-65.5,43.5,R2,R2,1.0,centroid,"POLYGON ((-65 43, -65 44, -66 44, -66 43, -65 ..."
3,-81.0,-80.0,44.0,45.0,-80.5,44.5,R3,R3,1.0,centroid,"POLYGON ((-80 44, -80 45, -81 45, -81 44, -80 ..."
4,-80.0,-79.0,44.0,45.0,-79.5,44.5,R4,R4,1.0,centroid,"POLYGON ((-79 44, -79 45, -80 45, -80 44, -79 ..."


In [14]:
# =============================================================================
# Build CANOE Region table from geospatial basemap
# =============================================================================

region_table = (
    regions[["region"]]
    .drop_duplicates()
    .sort_values("region", key=lambda s: s.str.extract(r"R(\d+)")[0].astype(int))
    .reset_index(drop=True)
)

region_table["notes"] = (
    "1 degree CANOE geospatial basemap region retained by centroid method"
)

print(f"New Region table rows: {len(region_table):,}")
print(f"Unique regions: {region_table['region'].nunique():,}")

display(region_table.head())
display(region_table.tail())

New Region table rows: 1,692
Unique regions: 1,692


,region,notes
0,R0,1 degree CANOE geospatial basemap region retai...
1,R1,1 degree CANOE geospatial basemap region retai...
2,R2,1 degree CANOE geospatial basemap region retai...
3,R3,1 degree CANOE geospatial basemap region retai...
4,R4,1 degree CANOE geospatial basemap region retai...


,region,notes
1687,R1687,1 degree CANOE geospatial basemap region retai...
1688,R1688,1 degree CANOE geospatial basemap region retai...
1689,R1689,1 degree CANOE geospatial basemap region retai...
1690,R1690,1 degree CANOE geospatial basemap region retai...
1691,R1691,1 degree CANOE geospatial basemap region retai...


In [15]:
# =============================================================================
# Validate geospatial Region table
# =============================================================================

assert "R-999" not in set(region_table["region"]), "R-999 should not be a real region."

assert (
    len(region_table) == region_table["region"].nunique()
), "Duplicate region IDs found."

assert (
    len(region_table) == len(regions)
), "Region table row count does not match centroid basemap row count."

print("Region table validated.")

Region table validated.


In [16]:
# =============================================================================
# Validate 1-degree centroid coordinate convention
# =============================================================================

resolution = regions["resolution_deg"].unique()

print(f"Resolution values: {resolution}")

assert len(resolution) == 1, "Multiple resolutions found in regions table."

resolution_deg = float(resolution[0])

lon_offset = ((regions["lon"] - regions["lon_min"]) / resolution_deg).round(6)
lat_offset = ((regions["lat"] - regions["lat_min"]) / resolution_deg).round(6)

assert set(lon_offset.unique()) == {0.5}, "Longitude centroids are not centered within cells."
assert set(lat_offset.unique()) == {0.5}, "Latitude centroids are not centered within cells."

print("Centroid coordinate convention validated.")

Resolution values: [1.]
Centroid coordinate convention validated.


In [17]:
# =============================================================================
# Inspect weak road edge table
# =============================================================================

print(f"Weak road edge rows: {len(road_edges_weak):,}")
print(f"Columns: {list(road_edges_weak.columns)}")

display(
    road_edges_weak.head()
)

Weak road edge rows: 1,518
Columns: ['region_from', 'region_to', 'direction', 'has_road_from', 'has_road_to', 'has_road_connection', 'connection_method', 'region_pair', 'lon_from', 'lat_from', 'lon_to', 'lat_to', 'geometry']


,region_from,region_to,direction,has_road_from,has_road_to,has_road_connection,connection_method,region_pair,lon_from,lat_from,lon_to,lat_to,geometry
0,R0,R1,right,True,True,True,weak_presence_adjacency,R0-R1,-81.5,43.5,-80.5,43.5,"LINESTRING (-81.5 43.5, -80.5 43.5)"
1,R1,R3,up,True,True,True,weak_presence_adjacency,R1-R3,-80.5,43.5,-80.5,44.5,"LINESTRING (-80.5 43.5, -80.5 44.5)"
2,R1,R0,left,True,True,True,weak_presence_adjacency,R1-R0,-80.5,43.5,-81.5,43.5,"LINESTRING (-80.5 43.5, -81.5 43.5)"
3,R2,R8,up,True,True,True,weak_presence_adjacency,R2-R8,-65.5,43.5,-65.5,44.5,"LINESTRING (-65.5 43.5, -65.5 44.5)"
4,R3,R1,down,True,True,True,weak_presence_adjacency,R3-R1,-80.5,44.5,-80.5,43.5,"LINESTRING (-80.5 44.5, -80.5 43.5)"


In [18]:
# =============================================================================
# Build clean weak road-link table
# =============================================================================

road_links_weak = road_edges_weak.copy()

road_links_weak = road_links_weak.loc[
    road_links_weak["has_road_connection"] == True
].copy()

road_links_weak = road_links_weak[
    [
        "region_from",
        "region_to",
        "direction",
        "region_pair",
        "connection_method",
        "lon_from",
        "lat_from",
        "lon_to",
        "lat_to",
    ]
].copy()

road_links_weak = road_links_weak.drop_duplicates(
    subset=["region_from", "region_to"]
).reset_index(drop=True)

road_links_weak["canoe_region"] = road_links_weak["region_pair"]

print(f"Weak road links: {len(road_links_weak):,}")
print(f"Unique CANOE transport regions: {road_links_weak['canoe_region'].nunique():,}")

display(
    road_links_weak.head()
)

Weak road links: 1,518
Unique CANOE transport regions: 1,518


,region_from,region_to,direction,region_pair,connection_method,lon_from,lat_from,lon_to,lat_to,canoe_region
0,R0,R1,right,R0-R1,weak_presence_adjacency,-81.5,43.5,-80.5,43.5,R0-R1
1,R1,R3,up,R1-R3,weak_presence_adjacency,-80.5,43.5,-80.5,44.5,R1-R3
2,R1,R0,left,R1-R0,weak_presence_adjacency,-80.5,43.5,-81.5,43.5,R1-R0
3,R2,R8,up,R2-R8,weak_presence_adjacency,-65.5,43.5,-65.5,44.5,R2-R8
4,R3,R1,down,R3-R1,weak_presence_adjacency,-80.5,44.5,-80.5,43.5,R3-R1


In [19]:
# =============================================================================
# Validate weak road-link table
# =============================================================================

valid_regions = set(region_table["region"])

invalid_from = sorted(
    set(road_links_weak["region_from"]) - valid_regions
)

invalid_to = sorted(
    set(road_links_weak["region_to"]) - valid_regions
)

assert not invalid_from, f"Invalid region_from values found: {invalid_from[:10]}"
assert not invalid_to, f"Invalid region_to values found: {invalid_to[:10]}"

assert (
    road_links_weak["canoe_region"].str.contains("-", regex=False).all()
), "All CANOE transport regions should use region_from-region_to format."

assert (
    road_links_weak["canoe_region"].nunique() == len(road_links_weak)
), "Duplicate CANOE transport region IDs found."

print("Weak road-link table validated.")

Weak road-link table validated.


In [20]:
# =============================================================================
# Compute road-link distances
# =============================================================================

road_links_weak_gdf = gpd.GeoDataFrame(
    road_links_weak,
    geometry=gpd.points_from_xy(
        road_links_weak["lon_from"],
        road_links_weak["lat_from"],
    ),
    crs="EPSG:4326",
)

road_links_weak_to_gdf = gpd.GeoDataFrame(
    road_links_weak,
    geometry=gpd.points_from_xy(
        road_links_weak["lon_to"],
        road_links_weak["lat_to"],
    ),
    crs="EPSG:4326",
)

# Use a projected CRS for approximate metric distances.
road_links_weak_from_m = road_links_weak_gdf.to_crs("EPSG:3347")
road_links_weak_to_m = road_links_weak_to_gdf.to_crs("EPSG:3347")

road_links_weak["distance_km"] = (
    road_links_weak_from_m.geometry.distance(
        road_links_weak_to_m.geometry
    )
    / 1000
)

print("Weak road-link distance summary:")
display(
    road_links_weak["distance_km"].describe()
)

display(
    road_links_weak.head()
)

Weak road-link distance summary:


count    1518.000000
mean       86.075605
std        22.820870
min        41.485806
25%        67.088321
50%        77.563333
75%       109.776058
max       113.603222
Name: distance_km, dtype: float64

,region_from,region_to,direction,region_pair,connection_method,lon_from,lat_from,lon_to,lat_to,canoe_region,distance_km
0,R0,R1,right,R0-R1,weak_presence_adjacency,-81.5,43.5,-80.5,43.5,R0-R1,82.896715
1,R1,R3,up,R1-R3,weak_presence_adjacency,-80.5,43.5,-80.5,44.5,R1-R3,113.603222
2,R1,R0,left,R1-R0,weak_presence_adjacency,-80.5,43.5,-81.5,43.5,R1-R0,82.896715
3,R2,R8,up,R2-R8,weak_presence_adjacency,-65.5,43.5,-65.5,44.5,R2-R8,113.603222
4,R3,R1,down,R3-R1,weak_presence_adjacency,-80.5,44.5,-80.5,43.5,R3-R1,113.603222


In [21]:
# =============================================================================
# Inspect transport edge regions in baseline model
# =============================================================================

transport_regions = db["CostVariable"]["region"].unique()

transport_regions = pd.Series(
    transport_regions,
    name="region"
)

transport_regions = transport_regions[
    transport_regions.str.contains("-")
]

print(f"Transport edge regions: {len(transport_regions):,}")

display(
    transport_regions.head(20)
)

Transport edge regions: 8,774


0           R0-R1
1          R0-R11
2           R1-R0
3          R1-R12
4           R1-R2
5         R10-R21
6          R10-R9
7       R100-R101
8       R100-R118
9        R100-R85
10       R100-R99
11    R1000-R1001
12    R1000-R1032
13     R1000-R969
14     R1000-R999
15    R1001-R1000
16    R1001-R1002
17    R1001-R1033
18     R1001-R970
19    R1002-R1001
Name: region, dtype: object

In [22]:
# =============================================================================
# Replace baseline Region table with geospatial basemap regions
# =============================================================================

db_encoded = {
    table_name: df.copy()
    for table_name, df in db.items()
}

db_encoded["Region"] = region_table.copy()

print(f"Baseline Region rows: {len(db['Region']):,}")
print(f"Encoded Region rows: {len(db_encoded['Region']):,}")

display(
    db_encoded["Region"].head()
)

Baseline Region rows: 2,259
Encoded Region rows: 1,692


,region,notes
0,R0,1 degree CANOE geospatial basemap region retai...
1,R1,1 degree CANOE geospatial basemap region retai...
2,R2,1 degree CANOE geospatial basemap region retai...
3,R3,1 degree CANOE geospatial basemap region retai...
4,R4,1 degree CANOE geospatial basemap region retai...


In [23]:
# =============================================================================
# Define road transport technologies
# =============================================================================

truck_tech_specs = pd.DataFrame(
    [
        {
            "tech": "CO2_TRUCK",
            "input_comm": "co2",
            "output_comm": "co2",
            "description": "Road transport of carbon dioxide by truck",
        },
        {
            "tech": "H2_TRUCK",
            "input_comm": "h2",
            "output_comm": "h2",
            "description": "Road transport of hydrogen by truck",
        },
        {
            "tech": "GSL_TRUCK",
            "input_comm": "gsl",
            "output_comm": "gsl",
            "description": "Road transport of gasoline by truck",
        },
        {
            "tech": "METOH_TRUCK",
            "input_comm": "ch3oh",
            "output_comm": "ch3oh",
            "description": "Road transport of methanol by truck",
        },
    ]
)

display(truck_tech_specs)

,tech,input_comm,output_comm,description
0,CO2_TRUCK,co2,co2,Road transport of carbon dioxide by truck
1,H2_TRUCK,h2,h2,Road transport of hydrogen by truck
2,GSL_TRUCK,gsl,gsl,Road transport of gasoline by truck
3,METOH_TRUCK,ch3oh,ch3oh,Road transport of methanol by truck


In [24]:
# =============================================================================
# Build truck Technology rows
# =============================================================================

technology_template = (
    db["Technology"]
    .loc[
        db["Technology"]["tech"] == "H2_PIPE"
    ]
    .copy()
)

truck_technology = pd.concat(
    [
        technology_template.assign(
            tech=row.tech,
            description=row.description,
        )
        for row in truck_tech_specs.itertuples()
    ],
    ignore_index=True,
)

display(
    truck_technology
)

,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
0,CO2_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of carbon dioxide by truck,GEO001
1,H2_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of hydrogen by truck,GEO001
2,GSL_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of gasoline by truck,GEO001
3,METOH_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of methanol by truck,GEO001


In [25]:
# =============================================================================
# Add truck technologies to encoded database
# =============================================================================

technology_template = (
    db_encoded["Technology"]
    .loc[db_encoded["Technology"]["tech"] == "H2_PIPE"]
    .copy()
)

truck_technology = pd.concat(
    [
        technology_template.assign(
            tech=row.tech,
            description=row.description,
        )
        for row in truck_tech_specs.itertuples(index=False)
    ],
    ignore_index=True,
)

db_encoded["Technology"] = (
    pd.concat(
        [
            db_encoded["Technology"],
            truck_technology,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["tech", "data_id"],
        keep="last",
    )
)

assert set(truck_tech_specs["tech"]).issubset(
    set(db_encoded["Technology"]["tech"])
)

print(f"Technology rows: {len(db_encoded['Technology']):,}")
display(db_encoded["Technology"].sort_values("tech"))

Technology rows: 16


,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
2,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
7,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
12,CO2_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of carbon dioxide by truck,GEO001
0,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
5,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
11,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
10,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
9,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
14,GSL_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of gasoline by truck,GEO001


In [26]:
# =============================================================================
# Build truck Efficiency rows for weak road links
# =============================================================================

truck_efficiency_rows = []

for truck in truck_tech_specs.itertuples(index=False):
    df = pd.DataFrame(
        {
            "region": road_links_weak["canoe_region"],
            "input_comm": truck.input_comm,
            "tech": truck.tech,
            "vintage": 1,
            "output_comm": truck.output_comm,
            "efficiency": 1.0,
            "notes": "Existing weak road-connected transport link",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    truck_efficiency_rows.append(df)

truck_efficiency_weak = pd.concat(
    truck_efficiency_rows,
    ignore_index=True,
)

print(f"Truck Efficiency rows: {len(truck_efficiency_weak):,}")

display(
    truck_efficiency_weak.head()
)

Truck Efficiency rows: 6,072


,region,input_comm,tech,vintage,output_comm,efficiency,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001
1,R1-R3,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001
2,R1-R0,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001
3,R2-R8,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001
4,R3-R1,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001


In [27]:
# =============================================================================
# Add truck Efficiency rows to database
# =============================================================================

db_encoded["Efficiency"] = (
    pd.concat(
        [
            db_encoded["Efficiency"],
            truck_efficiency_weak,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "region",
            "input_comm",
            "tech",
            "vintage",
            "output_comm",
            "data_id",
        ],
        keep="last",
    )
)

print(f"Encoded Efficiency rows: {len(db_encoded['Efficiency']):,}")

Encoded Efficiency rows: 68,260


In [28]:
# =============================================================================
# Build placeholder truck CostVariable rows for weak road links
# =============================================================================

PLACEHOLDER_TRUCK_COST_PER_KM = 0.01
PLACEHOLDER_TRUCK_INTERCEPT_COST = 0.0

truck_costvariable_rows = []

for truck in truck_tech_specs.itertuples(index=False):
    df = pd.DataFrame(
        {
            "region": road_links_weak["canoe_region"],
            "period": 1,
            "tech": truck.tech,
            "vintage": 1,
            "cost": (
                PLACEHOLDER_TRUCK_INTERCEPT_COST
                + PLACEHOLDER_TRUCK_COST_PER_KM * road_links_weak["distance_km"]
            ),
            "units": "M$/unit",
            "notes": "Placeholder truck transport cost based on weak road-connected centroid distance",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    truck_costvariable_rows.append(df)

truck_costvariable_weak = pd.concat(
    truck_costvariable_rows,
    ignore_index=True,
)

print(f"Truck CostVariable rows: {len(truck_costvariable_weak):,}")

display(
    truck_costvariable_weak.head()
)

Truck CostVariable rows: 6,072


,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,1,CO2_TRUCK,1,0.828967,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001
1,R1-R3,1,CO2_TRUCK,1,1.136032,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001
2,R1-R0,1,CO2_TRUCK,1,0.828967,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001
3,R2-R8,1,CO2_TRUCK,1,1.136032,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001
4,R3-R1,1,CO2_TRUCK,1,1.136032,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001


In [29]:
# =============================================================================
# Validate truck CostVariable rows
# =============================================================================

expected_costvariable_rows = len(road_links_weak) * len(truck_tech_specs)

assert len(truck_costvariable_weak) == expected_costvariable_rows

assert (
    truck_costvariable_weak[
        ["region", "period", "tech", "vintage", "data_id"]
    ].duplicated().sum() == 0
), "Duplicate CostVariable primary keys found."

assert (
    truck_costvariable_weak["cost"].notna().all()
), "Truck CostVariable contains missing costs."

assert (
    truck_costvariable_weak["cost"] >= 0
).all(), "Truck CostVariable contains negative costs."

print("Truck CostVariable rows validated.")

Truck CostVariable rows validated.


In [30]:
# =============================================================================
# Add truck CostVariable rows to database
# =============================================================================

db_encoded["CostVariable"] = (
    pd.concat(
        [
            db_encoded["CostVariable"],
            truck_costvariable_weak,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "region",
            "period",
            "tech",
            "vintage",
            "data_id",
        ],
        keep="last",
    )
)

print(f"Encoded CostVariable rows: {len(db_encoded['CostVariable']):,}")

Encoded CostVariable rows: 56,559


In [31]:
# =============================================================================
# Helper: filter node and edge regions by valid endpoints
# =============================================================================

def filter_valid_node_and_edge_regions(
    df: pd.DataFrame,
    valid_regions: set[str],
    region_col: str = "region",
) -> pd.DataFrame:
    """
    Keep rows whose region is either:
    1. a valid node region, e.g. R10, or
    2. a valid edge pseudo-region, e.g. R10-R11, where both endpoints are valid.
    """

    if region_col not in df.columns:
        return df.copy()

    out = df.copy()

    region_values = out[region_col].astype(str)

    node_mask = region_values.isin(valid_regions)

    edge_mask = region_values.str.contains("-", regex=False)

    valid_edge_mask = pd.Series(
        False,
        index=out.index,
    )

    if edge_mask.any():
        edge_parts = (
            region_values.loc[edge_mask]
            .str.split("-", n=1, expand=True)
        )

        valid_edge_mask.loc[edge_mask] = (
            edge_parts.iloc[:, 0].isin(valid_regions).values
            & edge_parts.iloc[:, 1].isin(valid_regions).values
        )

    keep_mask = node_mask | valid_edge_mask

    return (
        out.loc[keep_mask]
        .copy()
        .reset_index(drop=True)
    )

In [32]:
# =============================================================================
# Check node-region references after Region table replacement
# =============================================================================

valid_node_regions = set(db_encoded["Region"]["region"])

region_reference_report = []

for table_name, df in db_encoded.items():
    if "region" not in df.columns:
        continue

    region_values = df["region"].dropna().astype(str)

    # Node regions are plain R-number labels.
    # Edge pseudo-regions use R-from-R-to and are checked separately later.
    node_region_values = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(
        set(node_region_values) - valid_node_regions
    )

    region_reference_report.append(
        {
            "table": table_name,
            "rows": len(df),
            "node_region_values": node_region_values.nunique(),
            "invalid_node_regions": len(invalid_node_regions),
            "example_invalid_regions": invalid_node_regions[:10],
        }
    )

region_reference_report = pd.DataFrame(region_reference_report)

display(
    region_reference_report.sort_values(
        "invalid_node_regions",
        ascending=False,
    )
)

,table,rows,node_region_values,invalid_node_regions,example_invalid_regions
8,CostVariable,56559,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
7,CostInvest,4518,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
12,Efficiency,68260,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
17,ETLSegment,193552,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
33,LimitCapacity,4518,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
...,...,...,...,...,...
49,StorageDuration,0,0,0,[]
48,ReserveCapacityDerate,0,0,0,[]
55,OutputRetiredCapacity,0,0,0,[]
58,OutputStorageLevel,0,0,0,[]


In [33]:
# =============================================================================
# Filter edge-region tables to valid geospatial endpointsfilter_valid_node
#
# Remove inherited transport links whose endpoint regions are no longer
# present in the geospatial Region table. For edge pseudo-regions (Ri-Rj),
# retain only those links whose two endpoint regions are both valid.
# This preserves existing candidate pipeline links and newly added truck
# transport links while eliminating legacy connections to removed regions.
# =============================================================================
edge_region_tables = [
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]

for table_name in edge_region_tables:
    before_rows = len(db_encoded[table_name])

    db_encoded[table_name] = filter_valid_node_and_edge_regions(
        db_encoded[table_name],
        valid_regions=valid_node_regions,
    )

    after_rows = len(db_encoded[table_name])

    print(f"{table_name}: {before_rows:,} → {after_rows:,} ({after_rows - before_rows:+,})")

Efficiency: 68,260 → 52,560 (-15,700)
CostVariable: 56,559 → 43,896 (-12,663)
CostInvest: 4,518 → 3,384 (-1,134)
ETLSegment: 193,552 → 144,656 (-48,896)


In [34]:
# =============================================================================
# Build zero CostInvest rows for truck road links
# =============================================================================

truck_costinvest_rows = []

for truck in truck_tech_specs.itertuples(index=False):
    df = pd.DataFrame(
        {
            "region": road_links_weak["canoe_region"],
            "tech": truck.tech,
            "vintage": 1,
            "cost": 0.0,
            "units": "M$/unit",
            "notes": "Existing road transport link; no road construction investment encoded",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    truck_costinvest_rows.append(df)

truck_costinvest_weak = pd.concat(
    truck_costinvest_rows,
    ignore_index=True,
)

print(f"Truck CostInvest rows: {len(truck_costinvest_weak):,}")

display(
    truck_costinvest_weak.head()
)

Truck CostInvest rows: 6,072


,region,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001
1,R1-R3,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001
2,R1-R0,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001
3,R2-R8,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001
4,R3-R1,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001


In [35]:
# =============================================================================
# Validate truck CostInvest rows
# =============================================================================

expected_costinvest_rows = len(road_links_weak) * len(truck_tech_specs)

assert len(truck_costinvest_weak) == expected_costinvest_rows

assert (
    truck_costinvest_weak[
        ["region", "tech", "vintage", "data_id"]
    ].duplicated().sum() == 0
), "Duplicate CostInvest primary keys found."

assert (
    truck_costinvest_weak["cost"] == 0
).all(), "Truck CostInvest rows should have zero investment cost."

print("Truck CostInvest rows validated.")

Truck CostInvest rows validated.


In [36]:
# =============================================================================
# Add truck CostInvest rows to database
# =============================================================================

db_encoded["CostInvest"] = (
    pd.concat(
        [
            db_encoded["CostInvest"],
            truck_costinvest_weak,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "region",
            "tech",
            "vintage",
            "data_id",
        ],
        keep="last",
    )
)

print(f"Encoded CostInvest rows: {len(db_encoded['CostInvest']):,}")

Encoded CostInvest rows: 9,456


In [37]:
# =============================================================================
# Summarize encoded database changes
# =============================================================================

summary_rows = []

for table in [
    "Region",
    "Technology",
    "Efficiency",
    "CostVariable",
    "CostInvest",
]:
    summary_rows.append(
        {
            "table": table,
            "baseline_rows": len(db[table]),
            "encoded_rows": len(db_encoded[table]),
            "delta_rows": len(db_encoded[table]) - len(db[table]),
        }
    )

encoding_summary = pd.DataFrame(summary_rows)

display(encoding_summary)

,table,baseline_rows,encoded_rows,delta_rows
0,Region,2259,1692,-567
1,Technology,12,16,4
2,Efficiency,62188,52560,-9628
3,CostVariable,50487,43896,-6591
4,CostInvest,4518,9456,4938


## Reconcile inherited baseline tables with geospatial regions

The baseline SQLite database contains a complete runnable CANOE test model, but several inherited tables were generated using the old synthetic site aggregation workflow. Since the Region table has now been replaced by the geospatial 1° basemap regions, inherited node-based tables may contain references to regions that no longer exist.

The next cells identify and filter or rebuild these tables so that all node-region references are consistent with the geospatial Region table.

In [38]:
# =============================================================================
# Helper: filter tables to valid geospatial node regions
# =============================================================================

def filter_valid_regions(
    df: pd.DataFrame,
    valid_regions: set[str],
    region_col: str = "region",
    keep_edge_regions: bool = True,
) -> pd.DataFrame:
    """
    Filter rows in a CANOE table to valid geospatial node regions.

    Parameters
    ----------
    df
        Input CANOE table.
    valid_regions
        Set of valid node-region IDs from the geospatial Region table.
    region_col
        Name of the region column.
    keep_edge_regions
        If True, keep pseudo-regions such as R0-R1.

    Returns
    -------
    pd.DataFrame
        Filtered table.
    """

    if region_col not in df.columns:
        return df.copy()

    region_values = df[region_col].astype(str)

    node_mask = region_values.isin(valid_regions)

    if keep_edge_regions:
        edge_mask = region_values.str.contains("-", regex=False)
        keep_mask = node_mask | edge_mask
    else:
        keep_mask = node_mask

    return df.loc[keep_mask].copy().reset_index(drop=True)

In [39]:
# =============================================================================
# Load old model site dictionary for comparison
# =============================================================================

OLD_SITE_DICT_PATH = PROJECT_ROOT / "sites_dict_1.csv"

old_sites = pd.read_csv(OLD_SITE_DICT_PATH)

print(f"Old site dictionary rows: {len(old_sites):,}")
print(f"New geospatial regions: {len(regions):,}")

display(
    old_sites.head()
)

display(
    old_sites.columns
)

Old site dictionary rows: 2,259
New geospatial regions: 1,692


,lon,lat,LCOE,max_elc,demand,co2,region,up_id,down_id,right_id,left_id,right_distance,left_distance,up_distance,down_distance,site_id,co2_cost
0,-141,60,0.056970,9.904231e+05,0.0,0.0,R0,R1,R-999,R11,R-999,55.799470,NaN,111.420728,NaN,R0,50
1,-141,61,0.100296,1.805530e+06,0.0,0.0,R1,R2,R0,R12,R-999,54.106953,NaN,111.437373,111.420728,R1,50
2,-141,62,0.138735,9.350884e+05,0.0,0.0,R2,R3,R1,R13,R-999,52.397727,NaN,111.453649,111.437373,R2,50
3,-141,63,0.120369,1.248552e+06,0.0,0.0,R3,R4,R2,R14,R-999,50.672313,NaN,111.469535,111.453649,R3,50
4,-141,64,0.080409,1.285275e+06,0.0,0.0,R4,R5,R3,R15,R-999,48.931240,NaN,111.485013,111.469535,R4,50


Index(['lon', 'lat', 'LCOE', 'max_elc', 'demand', 'co2', 'region', 'up_id',
       'down_id', 'right_id', 'left_id', 'right_distance', 'left_distance',
       'up_distance', 'down_distance', 'site_id', 'co2_cost'],
      dtype='object')

In [40]:
# =============================================================================
# Compare old and new region ID coverage
# =============================================================================

old_site_regions = set(old_sites["site_id"])
new_region_ids = set(region_table["region"])

shared_regions = old_site_regions & new_region_ids
old_only_regions = old_site_regions - new_region_ids
new_only_regions = new_region_ids - old_site_regions

print(f"Shared regions: {len(shared_regions):,}")
print(f"Old-only regions: {len(old_only_regions):,}")
print(f"New-only regions: {len(new_only_regions):,}")

print("\nExample old-only regions:")
print(sorted(old_only_regions, key=lambda x: int(x.replace("R", "")))[:20])

print("\nExample new-only regions:")
print(sorted(new_only_regions, key=lambda x: int(x.replace("R", "")))[:20])

Shared regions: 1,692
Old-only regions: 567
New-only regions: 0

Example old-only regions:
['R1692', 'R1693', 'R1694', 'R1695', 'R1696', 'R1697', 'R1698', 'R1699', 'R1700', 'R1701', 'R1702', 'R1703', 'R1704', 'R1705', 'R1706', 'R1707', 'R1708', 'R1709', 'R1710', 'R1711']

Example new-only regions:
[]


In [41]:
# =============================================================================
# Filter inherited node-region tables to geospatial Region table
# =============================================================================

tables_to_filter_by_node_region = [
    "Demand",
    "LimitCapacity",
    "LimitTechInputSplitAnnual",
    "CostInvest",
    "CostVariable",
    "Efficiency",
]

for table_name in tables_to_filter_by_node_region:
    before_rows = len(db_encoded[table_name])

    db_encoded[table_name] = filter_valid_regions(
        db_encoded[table_name],
        valid_regions=valid_node_regions,
        keep_edge_regions=True,
    )

    after_rows = len(db_encoded[table_name])

    print(
        f"{table_name}: {before_rows:,} → {after_rows:,} "
        f"({after_rows - before_rows:+,})"
    )

Demand: 123 → 86 (-37)
LimitCapacity: 4,518 → 3,384 (-1,134)
LimitTechInputSplitAnnual: 11,295 → 8,460 (-2,835)
CostInvest: 9,456 → 9,456 (+0)
CostVariable: 43,896 → 43,896 (+0)
Efficiency: 52,560 → 52,560 (+0)


In [42]:
# =============================================================================
# Re-check node-region references after filtering
# =============================================================================

region_reference_report_filtered = []

for table_name, df in db_encoded.items():
    if "region" not in df.columns:
        continue

    region_values = df["region"].dropna().astype(str)

    node_region_values = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(
        set(node_region_values) - valid_node_regions
    )

    region_reference_report_filtered.append(
        {
            "table": table_name,
            "rows": len(df),
            "node_region_values": node_region_values.nunique(),
            "invalid_node_regions": len(invalid_node_regions),
            "example_invalid_regions": invalid_node_regions[:10],
        }
    )

region_reference_report_filtered = pd.DataFrame(
    region_reference_report_filtered
)

display(
    region_reference_report_filtered.sort_values(
        "invalid_node_regions",
        ascending=False,
    )
)

,table,rows,node_region_values,invalid_node_regions,example_invalid_regions
60,OutputCost,2187,613,165,"[R1716, R1719, R1720, R1721, R1722, R1723, R17..."
56,OutputFlowIn,1608,155,47,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
57,OutputFlowOut,1608,155,47,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
53,OutputNetCapacity,1693,154,46,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
54,OutputBuiltCapacity,1693,154,46,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
...,...,...,...,...,...
49,StorageDuration,0,0,0,[]
48,ReserveCapacityDerate,0,0,0,[]
55,OutputRetiredCapacity,0,0,0,[]
58,OutputStorageLevel,0,0,0,[]


In [43]:
# =============================================================================
# Inspect ETLSegment technologies
# =============================================================================

etl_tech_counts = (
    db_encoded["ETLSegment"]
    .groupby("tech_or_group")
    .size()
    .reset_index(name="rows")
    .sort_values(
        "rows",
        ascending=False,
    )
)

display(
    etl_tech_counts
)

,tech_or_group,rows
0,CO2_PIPE,26224
1,ELC_TRANS,26224
2,GSL_PIPE,26224
4,H2_PIPE,26224
5,METOH_PIPE,26224
3,GSL_PLANT,6768
6,METOH_PLANT,6768


In [44]:
# =============================================================================
# Characterize ETLSegment region types
# =============================================================================

etl = db_encoded["ETLSegment"].copy()

etl["is_edge_region"] = (
    etl["region"]
    .astype(str)
    .str.contains("-", regex=False)
)

etl_region_summary = (
    etl
    .groupby(
        ["tech_or_group", "is_edge_region"]
    )
    .size()
    .reset_index(name="rows")
    .sort_values(
        ["tech_or_group", "is_edge_region"]
    )
)

display(etl_region_summary)

,tech_or_group,is_edge_region,rows
0,CO2_PIPE,True,26224
1,ELC_TRANS,True,26224
2,GSL_PIPE,True,26224
3,GSL_PLANT,False,6768
4,H2_PIPE,True,26224
5,METOH_PIPE,True,26224
6,METOH_PLANT,False,6768


In [45]:
# =============================================================================
# Characterize Efficiency region types
# =============================================================================

eff = db_encoded["Efficiency"].copy()

eff["is_edge_region"] = (
    eff["region"]
    .astype(str)
    .str.contains("-", regex=False)
)

eff_region_summary = (
    eff
    .groupby(
        ["tech", "is_edge_region"]
    )
    .size()
    .reset_index(name="rows")
    .sort_values(
        ["tech", "is_edge_region"]
    )
)

display(eff_region_summary)

,tech,is_edge_region,rows
0,CO2_CAP,False,1692
1,CO2_PIPE,True,6556
2,CO2_TRUCK,True,1518
3,ELC_GEN,False,1692
4,ELC_TRANS,True,6556
5,GSL_BACKUP,False,86
6,GSL_DEMAND,False,86
7,GSL_PIPE,True,6556
8,GSL_PLANT,False,3384
9,GSL_TRUCK,True,1518


In [46]:
# =============================================================================
# Check ETLSegment node-region validity
# =============================================================================

etl = db_encoded["ETLSegment"].copy()

etl["is_edge_region"] = (
    etl["region"]
    .astype(str)
    .str.contains("-", regex=False)
)

node_etl = etl.loc[
    ~etl["is_edge_region"]
].copy()

invalid_node_etl = (
    ~node_etl["region"].isin(valid_node_regions)
)

print(f"Node ETLSegment rows: {len(node_etl):,}")
print(f"Invalid node ETLSegment rows: {invalid_node_etl.sum():,}")
print(f"Valid node ETLSegment rows: {(~invalid_node_etl).sum():,}")

display(
    node_etl.loc[
        invalid_node_etl
    ].head()
)

Node ETLSegment rows: 13,536
Invalid node ETLSegment rows: 0
Valid node ETLSegment rows: 13,536


,region,tech_or_group,segment,cap_lower,cap_upper,cost_lower,cost_upper,data_id,is_edge_region


In [47]:
# =============================================================================
# Filter ETLSegment node-region rows to geospatial Region table
# =============================================================================

before_rows = len(db_encoded["ETLSegment"])

db_encoded["ETLSegment"] = filter_valid_regions(
    db_encoded["ETLSegment"],
    valid_regions=valid_node_regions,
    keep_edge_regions=True,
)

after_rows = len(db_encoded["ETLSegment"])

print(
    f"ETLSegment: {before_rows:,} → {after_rows:,} "
    f"({after_rows - before_rows:+,})"
)

ETLSegment: 144,656 → 144,656 (+0)


In [48]:
# =============================================================================
# Final node-region consistency check
# =============================================================================

region_reference_report_final = []

for table_name, df in db_encoded.items():

    if "region" not in df.columns:
        continue

    region_values = df["region"].dropna().astype(str)

    node_region_values = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(
        set(node_region_values) - valid_node_regions
    )

    region_reference_report_final.append(
        {
            "table": table_name,
            "rows": len(df),
            "node_region_values": node_region_values.nunique(),
            "invalid_node_regions": len(invalid_node_regions),
            "example_invalid_regions": invalid_node_regions[:10],
        }
    )

region_reference_report_final = pd.DataFrame(
    region_reference_report_final
)

display(
    region_reference_report_final.sort_values(
        "invalid_node_regions",
        ascending=False,
    )
)

,table,rows,node_region_values,invalid_node_regions,example_invalid_regions
60,OutputCost,2187,613,165,"[R1716, R1719, R1720, R1721, R1722, R1723, R17..."
56,OutputFlowIn,1608,155,47,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
57,OutputFlowOut,1608,155,47,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
53,OutputNetCapacity,1693,154,46,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
54,OutputBuiltCapacity,1693,154,46,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
...,...,...,...,...,...
49,StorageDuration,0,0,0,[]
48,ReserveCapacityDerate,0,0,0,[]
55,OutputRetiredCapacity,0,0,0,[]
58,OutputStorageLevel,0,0,0,[]


In [49]:
# =============================================================================
# Clear solver output tables
# =============================================================================

output_tables = [
    table_name
    for table_name in db_encoded
    if table_name.startswith("Output")
]

for table_name in output_tables:

    before_rows = len(db_encoded[table_name])

    db_encoded[table_name] = (
        db_encoded[table_name]
        .iloc[0:0]
        .copy()
    )

    print(
        f"{table_name}: {before_rows:,} → {len(db_encoded[table_name]):,}"
    )

OutputCurtailment: 0 → 0
OutputNetCapacity: 1,693 → 0
OutputBuiltCapacity: 1,693 → 0
OutputRetiredCapacity: 0 → 0
OutputFlowIn: 1,608 → 0
OutputFlowOut: 1,608 → 0
OutputStorageLevel: 0 → 0
OutputDualVariable: 0 → 0
OutputObjective: 1 → 0
OutputEmission: 0 → 0
OutputCost: 2,187 → 0


In [50]:
# =============================================================================
# Re-run final node-region consistency check after clearing outputs
# =============================================================================

region_reference_report_final = []

for table_name, df in db_encoded.items():

    if "region" not in df.columns:
        continue

    region_values = df["region"].dropna().astype(str)

    node_region_values = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(
        set(node_region_values) - valid_node_regions
    )

    region_reference_report_final.append(
        {
            "table": table_name,
            "rows": len(df),
            "node_region_values": node_region_values.nunique(),
            "invalid_node_regions": len(invalid_node_regions),
            "example_invalid_regions": invalid_node_regions[:10],
        }
    )

region_reference_report_final = pd.DataFrame(
    region_reference_report_final
)

display(
    region_reference_report_final.sort_values(
        "invalid_node_regions",
        ascending=False,
    )
)

assert (
    region_reference_report_final["invalid_node_regions"] == 0
).all(), "Invalid node-region references remain."

print("All node-region references are valid.")

,table,rows,node_region_values,invalid_node_regions,example_invalid_regions
0,CapacityCredit,0,0,0,[]
1,CapacityFactorProcess,0,0,0,[]
2,CapacityFactorTech,0,0,0,[]
3,CapacityToActivity,0,0,0,[]
4,ConstructionInput,0,0,0,[]
...,...,...,...,...,...
56,OutputFlowIn,0,0,0,[]
57,OutputFlowOut,0,0,0,[]
58,OutputStorageLevel,0,0,0,[]
59,OutputEmission,0,0,0,[]


All node-region references are valid.


In [51]:
# =============================================================================
# Final encoded database summary
# =============================================================================

final_summary = []

for table_name, df in db_encoded.items():

    final_summary.append(
        {
            "table": table_name,
            "rows": len(df),
            "columns": len(df.columns),
        }
    )

final_summary = (
    pd.DataFrame(final_summary)
    .sort_values(
        "table"
    )
    .reset_index(drop=True)
)

display(final_summary)

,table,rows,columns
0,CapacityCredit,0,13
1,CapacityFactorProcess,0,15
2,CapacityFactorTech,0,14
3,CapacityToActivity,0,11
4,Commodity,7,4
...,...,...,...
80,Technology,16,15
81,TechnologyType,4,2
82,TimePeriod,2,3
83,TimePeriodType,2,2


In [52]:
# =============================================================================
# Create fresh weak-road SQLite database
# =============================================================================

if OUTPUT_SQLITE_WEAK_PATH.exists():
    OUTPUT_SQLITE_WEAK_PATH.unlink()

db_mgmt.convert_sql_to_sqlite(
    RAW_SCHEMA_PATH,
    OUTPUT_SQLITE_WEAK_PATH,
)

print(
    f"Created {OUTPUT_SQLITE_WEAK_PATH.name}"
)

Created CANOE_geospatial_1deg_roads_weak.sqlite


In [53]:
# =============================================================================
# Write encoded tables to SQLite
# =============================================================================

db_mgmt.update_sqlite(
    OUTPUT_SQLITE_WEAK_PATH,
    db_encoded,
)

print(
    f"Wrote {len(db_encoded)} tables to {OUTPUT_SQLITE_WEAK_PATH.name}"
)

Wrote 85 tables to CANOE_geospatial_1deg_roads_weak.sqlite


In [54]:
# =============================================================================
# Verify exported SQLite database
# =============================================================================

db_test = db_mgmt.sqlite_to_dfs(
    OUTPUT_SQLITE_WEAK_PATH
)

print(
    f"Exported tables: {len(db_test)}"
)

for table_name in [
    "Region",
    "Technology",
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:
    print(
        f"{table_name}: {len(db_test[table_name]):,}"
    )

Exported tables: 85
Region: 1,692
Technology: 16
Efficiency: 52,560
CostVariable: 43,896
CostInvest: 9,456
ETLSegment: 144,656


In [55]:
# =============================================================================
# Check truck technologies in working and exported databases
# =============================================================================

truck_tech_names = set(truck_tech_specs["tech"])

print("In db_encoded:")
print(
    sorted(
        truck_tech_names
        & set(db_encoded["Technology"]["tech"])
    )
)

print("\nIn exported SQLite:")
print(
    sorted(
        truck_tech_names
        & set(db_test["Technology"]["tech"])
    )
)

In db_encoded:
['CO2_TRUCK', 'GSL_TRUCK', 'H2_TRUCK', 'METOH_TRUCK']

In exported SQLite:
['CO2_TRUCK', 'GSL_TRUCK', 'H2_TRUCK', 'METOH_TRUCK']


In [56]:
db_mgmt.update_sqlite(
    OUTPUT_SQLITE_WEAK_PATH,
    db_encoded,
)

In [57]:
db_test = db_mgmt.sqlite_to_dfs(
    OUTPUT_SQLITE_WEAK_PATH
)

print(
    f"Technology: {len(db_test['Technology']):,}"
)

display(
    db_test["Technology"]
    .sort_values("tech")
)

Technology: 16


,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
2,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
7,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
12,CO2_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of carbon dioxide by truck,GEO001
0,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
5,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
11,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
10,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
9,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
14,GSL_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of gasoline by truck,GEO001


In [58]:
# =============================================================================
# Validate exported truck technologies and rows
# =============================================================================

truck_techs = set(truck_tech_specs["tech"])
expected_truck_rows = len(road_links_weak) * len(truck_tech_specs)

assert truck_techs.issubset(set(db_test["Technology"]["tech"]))

assert db_test["Efficiency"]["tech"].isin(truck_techs).sum() == expected_truck_rows
assert db_test["CostVariable"]["tech"].isin(truck_techs).sum() == expected_truck_rows
assert db_test["CostInvest"]["tech"].isin(truck_techs).sum() == expected_truck_rows

print("Truck technology rows validated.")

Truck technology rows validated.


In [59]:
# =============================================================================
# Validate exported node-region references
# =============================================================================

valid_node_regions = set(db_test["Region"]["region"])

for table_name, df in db_test.items():

    if "region" not in df.columns:
        continue

    region_values = df["region"].dropna().astype(str)
    node_regions = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(set(node_regions) - valid_node_regions)

    assert not invalid_node_regions, (
        f"{table_name} has invalid node regions: {invalid_node_regions[:10]}"
    )

print("Node-region references validated.")

Node-region references validated.


In [60]:
# =============================================================================
# Validate output tables are empty
# =============================================================================

for table_name, df in db_test.items():
    if table_name.startswith("Output"):
        assert len(df) == 0, f"{table_name} is not empty."

print("Output tables validated.")

Output tables validated.


In [61]:
# =============================================================================
# Validate output tables are empty
# =============================================================================

for table_name, df in db_test.items():
    if table_name.startswith("Output"):
        assert len(df) == 0, f"{table_name} is not empty."

print("Output tables validated.")

Output tables validated.


In [62]:
# =============================================================================
# Notebook 7 final export validation
# =============================================================================

print("Notebook 7 export validated.")
print(f"Output database: {OUTPUT_SQLITE_WEAK_PATH}")

Notebook 7 export validated.
Output database: c:\Users\aviga\Research\repos\temoa_geospace\data_files\processed\schema\CANOE_geospatial_1deg_roads_weak.sqlite


The sqlite db is now constructed. Next Cells will refer to model debugging and db structure.

In [63]:
# =============================================================================
# Validate edge-region endpoints
# =============================================================================

for table_name, df in db_encoded.items():

    if "region" not in df.columns:
        continue

    edge_values = (
        df["region"]
        .dropna()
        .astype(str)
    )

    edge_values = edge_values[
        edge_values.str.contains("-", regex=False)
    ]

    if len(edge_values) == 0:
        continue

    edge_parts = edge_values.str.split("-", n=1, expand=True)

    invalid_edges = edge_values.loc[
        ~(
            edge_parts.iloc[:, 0].isin(valid_node_regions).values
            &
            edge_parts.iloc[:, 1].isin(valid_node_regions).values
        )
    ]

    assert invalid_edges.empty, (
        f"{table_name} has invalid edge endpoints:\n"
        f"{invalid_edges.head()}"
    )

print("Edge-region endpoints validated.")

Edge-region endpoints validated.
